# Nirapod — food segmentation (PUST Cafeteria Food → YOLO-seg → ONNX)

Trains a small instance-segmentation model on the **PUST Cafeteria Food Image Dataset** (Bangladeshi cafeteria
meal platters, YOLO-seg polygon labels) and exports it for **onnxruntime-web** in the Nirapod app.

**Kaggle settings:** Accelerator **GPU T4 x2** (the notebook uses one GPU), Internet **On**, Persistence off.
Add the dataset with **Add Input**, set `INPUT_DIR` below, then **Save Version → Save & Run All (Commit)**.
Training is time-boxed (`TRAIN_HOURS`), so the whole run finishes in well under 2 hours.

**Outputs in `/kaggle/working/`:**
`food_seg.onnx`, `food_seg.json` (copy both to `public/models/` in the app), `metrics.json`, `per_class_ap.csv`,
`confusion_matrix.csv`, `confusion_matrix*.png`, `sample_predictions.jpg`, `samples/`, `class_counts.png`,
`best.pt`, and the full Ultralytics run in `runs/`.

**Dataset:** PUST Cafeteria Food Image Dataset, Mendeley Data, v2, DOI
[10.17632/fn6yhzjz83.2](https://doi.org/10.17632/fn6yhzjz83.2). Check its licence on the Mendeley page and
cite it wherever results are shown. The dataset authors note that billing, portion or calorie estimation needs
additional validated information — the app only uses the model to recognise foods, and shows a signal, not grams.

In [ ]:
# ---------------------------------------------------------------- settings (edit these) ----
import os, time
RUN_STARTED = time.time()

INPUT_DIR = os.environ.get('FOOD_SEG_INPUT', '/kaggle/input/pust-cafeteria-food')  # ← your dataset slug
WORK_DIR = os.environ.get('FOOD_SEG_WORK', '/kaggle/working')

# Small segmentation model. YOLO11n-seg is used because its ONNX output layout (boxes + 32 mask coefficients,
# plus mask prototypes) is the one the app's browser decoder expects; 'yolov8n-seg.pt' has the same layout.
MODEL_WEIGHTS = 'yolo11n-seg.pt'
FALLBACK_WEIGHTS = ['yolov8n-seg.pt']

TRAIN_HOURS = float(os.environ.get('FOOD_SEG_HOURS', 1.25))  # training time budget; Ultralytics stops and finalises at it
IMGSZ = 640
BATCH = 16
PATIENCE = 40              # stop early if mask mAP has not improved for this many epochs
SEED = 42
VAL_FRACTION = 0.15        # only used if the dataset has no train/val/test split of its own
TEST_FRACTION = 0.15
LABEL_ROOT_OVERRIDE = None # set to a folder path if the automatic choice of label folder is wrong
MODEL_VERSION = '0.1.0'
CLEANUP = True             # delete the rebuilt dataset copy from /kaggle/working at the end

# For checking the notebook on a laptop/CPU: FOOD_SEG_SMOKE=1 trains 1 tiny epoch.
SMOKE_TEST = os.environ.get('FOOD_SEG_SMOKE') == '1'

In [ ]:
# ---------------------------------------------------------------- install + environment ----
import subprocess, sys
ULTRALYTICS_SPEC = 'ultralytics>=8.4,<8.5'   # tested with 8.4.215
try:
    import ultralytics
    ok = tuple(int(x) for x in ultralytics.__version__.split('.')[:2]) == (8, 4)
except ImportError:
    ok = False
if not ok:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', ULTRALYTICS_SPEC], check=True)
    for mod in [m for m in sys.modules if m == 'ultralytics' or m.startswith('ultralytics.')]:
        del sys.modules[mod]  # forget an older version imported above
for pkg in ('onnx', 'onnxslim', 'onnxruntime'):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

import json, random, re, shutil
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.utils import YAML  # Ultralytics 8.4 has its own YAML reader/writer (PyYAML not required)

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__,
      '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if DEVICE == 'cpu' and not SMOKE_TEST:
    print('WARNING: no GPU found. In Kaggle: Settings → Accelerator → GPU T4 x2.')

WORK = Path(WORK_DIR)
WORK.mkdir(parents=True, exist_ok=True)
random.seed(SEED)

## 1. Find the data
Looks for a `data.yaml` with segmentation labels first; otherwise finds every folder of YOLO-seg label files
(polygons — not the YOLO_detect box files), pairs them with images by file name, reads class names from a
yaml / classes.txt / COCO json, and rebuilds a clean dataset under `/kaggle/working/data/`.

In [ ]:
INPUT = Path(INPUT_DIR)
if not INPUT.exists():
    kaggle_inputs = sorted(Path('/kaggle/input').glob('*')) if Path('/kaggle/input').exists() else []
    print(f'INPUT_DIR {INPUT} not found. Available inputs: {[str(p) for p in kaggle_inputs]}')
    if len(kaggle_inputs) == 1:
        INPUT = kaggle_inputs[0]
        print(f'Using the only input: {INPUT}')
    else:
        raise FileNotFoundError('Set INPUT_DIR to your dataset folder under /kaggle/input/')

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLIT_ALIASES = {'train': 'train', 'training': 'train', 'val': 'val', 'valid': 'val', 'validation': 'val', 'test': 'test', 'testing': 'test'}

all_files = [p for p in INPUT.rglob('*') if p.is_file()]
images = [p for p in all_files if p.suffix.lower() in IMG_EXT]
txts = [p for p in all_files if p.suffix.lower() == '.txt']
print(f'Found under {INPUT}: {len(all_files)} files, {len(images)} images, {len(txts)} .txt files')
top = Counter(p.relative_to(INPUT).parts[0] if len(p.relative_to(INPUT).parts) > 1 else '.' for p in all_files)
print('Top-level folders:', dict(top))

def label_kind(path, max_lines=50):
    """'seg' (class + polygon), 'det' (class + 4 box numbers), or None."""
    kinds = Counter()
    try:
        for i, line in enumerate(path.read_text(errors='ignore').splitlines()):
            if i >= max_lines:
                break
            parts = line.split()
            if not parts:
                continue
            try:
                nums = [float(x) for x in parts]
            except ValueError:
                return None
            if not float(nums[0]).is_integer():
                return None
            coords = nums[1:]
            if len(coords) == 4:
                kinds['det'] += 1
            elif len(coords) >= 6 and len(coords) % 2 == 0 and all(-0.01 <= c <= 1.01 for c in coords):
                kinds['seg'] += 1
            else:
                kinds['other'] += 1
    except OSError:
        return None
    if not kinds:
        return 'empty'
    kind, n = kinds.most_common(1)[0]
    return kind if kind in ('seg', 'det') and n >= 0.8 * sum(kinds.values()) else None

def split_of(path):
    for part in reversed(path.parts):
        if part.lower() in SPLIT_ALIASES:
            return SPLIT_ALIASES[part.lower()]
    return None

def names_from_yaml(path):
    data = YAML.load(path) or {}
    names = data.get('names')
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=int)]
    return list(names) if isinstance(names, list) else None

# --- 1a. a ready-made data.yaml whose labels are segmentation polygons --------------------------------------
DATA_YAML, CLASS_NAMES, SPLITS = None, None, None
for yml in sorted(INPUT.rglob('*.y*ml')):
    try:
        cfg = YAML.load(yml) or {}
    except Exception:
        continue
    if not isinstance(cfg, dict) or 'names' not in cfg or 'train' not in cfg:
        continue
    base = Path(cfg.get('path') or yml.parent)
    if not base.is_absolute():
        base = (yml.parent / base).resolve()
    resolved = {}
    for split in ('train', 'val', 'test'):
        if not cfg.get(split):
            continue
        p = Path(cfg[split])
        for cand in (p if p.is_absolute() else base / p, yml.parent / p, yml.parent / str(cfg[split]).lstrip('./').lstrip('../')):
            if cand.exists():
                resolved[split] = cand.resolve()
                break
    if 'train' not in resolved or 'val' not in resolved:
        print(f'{yml}: names found but its train/val paths do not exist here — ignored')
        continue
    lbl_dir = Path(str(resolved['train']).replace(f'{os.sep}images', f'{os.sep}labels'))
    sample = list(lbl_dir.rglob('*.txt'))[:20] if lbl_dir.exists() else []
    kinds = Counter(label_kind(s) for s in sample)
    print(f'{yml}: splits {list(resolved)}; label kinds in {lbl_dir}: {dict(kinds)}')
    if kinds.get('seg', 0) >= max(1, 0.8 * len(sample)):
        CLASS_NAMES = names_from_yaml(yml)
        SPLITS = {k: str(v) for k, v in resolved.items()}
        DATA_YAML = yml
        print(f'→ Using {yml} as is ({len(CLASS_NAMES)} classes)')
        break

In [ ]:
# --- 1b. otherwise: find YOLO-seg label folders and rebuild a dataset --------------------------------------
def group_key(stem):
    """Augmented copies of one photo share a key, so they always land in the same split."""
    s = re.sub(r'(_(jpg|jpeg|png))?\.rf\.[0-9a-f]+$', '', stem, flags=re.I)  # Roboflow export names
    s = re.sub(r'([_\-. ](aug|augmented|flip|flipped|rot|rotated|bright|dark|noise|blur|crop|copy)[\w\-]*)$', '', s, flags=re.I)
    return s.lower()

if DATA_YAML is None:
    seg_dirs = defaultdict(list)
    det_dirs = Counter()
    for t in txts:
        kind = label_kind(t)
        if kind == 'seg':
            seg_dirs[t.parent].append(t)
        elif kind == 'det':
            det_dirs[t.parent] += 1
    if det_dirs:
        print('Ignoring box-only (YOLO_detect) label folders:', {str(k.relative_to(INPUT)): v for k, v in det_dirs.items()})
    if not seg_dirs:
        raise RuntimeError('No YOLO segmentation label files found. Check the dataset contents printed above.')

    # Group label folders into "label sets": the folder path with split names and 'labels' removed.
    def root_of(d):
        parts = [p for p in d.relative_to(INPUT).parts if p.lower() not in SPLIT_ALIASES and p.lower() != 'labels']
        return INPUT.joinpath(*parts) if parts else INPUT
    roots = defaultdict(list)
    for d, files in seg_dirs.items():
        roots[root_of(d)].extend(files)

    by_stem = defaultdict(list)
    for img in images:
        by_stem[img.stem].append(img)

    def pair(label):
        """Image for a label file: same stem, preferring the parallel images/ folder, else a unique match."""
        cands = by_stem.get(label.stem, [])
        if len(cands) == 1:
            return cands[0]
        parallel = Path(str(label.parent).replace(f'{os.sep}labels', f'{os.sep}images'))
        for c in cands:
            if c.parent == parallel:
                return c
        same_split = [c for c in cands if split_of(c) == split_of(label)]
        return same_split[0] if len(same_split) == 1 else None

    print('\nSegmentation label sets found:')
    summary = {}
    for root, files in roots.items():
        pairs = [(f, pair(f)) for f in files]
        ok = [(f, i) for f, i in pairs if i is not None]
        summary[root] = ok
        print(f'  {root.relative_to(INPUT) if root != INPUT else "."}: {len(files)} label files, {len(ok)} paired with images')
    chosen = Path(LABEL_ROOT_OVERRIDE) if LABEL_ROOT_OVERRIDE else max(summary, key=lambda r: len(summary[r]))
    pairs = summary[chosen]
    print(f'→ Using label set: {chosen} ({len(pairs)} image/label pairs). Set LABEL_ROOT_OVERRIDE to choose another.')

    # Class names: yaml → classes.txt / obj.names → COCO json categories.
    for search_root in (chosen, INPUT):
        for yml in sorted(search_root.rglob('*.y*ml')):
            try:
                CLASS_NAMES = names_from_yaml(yml)
            except Exception:
                CLASS_NAMES = None
            if CLASS_NAMES:
                print(f'Class names from {yml}')
                break
        if CLASS_NAMES:
            break
        for txt_name in ('classes.txt', 'obj.names', 'labels.txt'):
            hits = sorted(search_root.rglob(txt_name))
            if hits:
                CLASS_NAMES = [l.strip() for l in hits[0].read_text(errors='ignore').splitlines() if l.strip()]
                print(f'Class names from {hits[0]}')
                break
        if CLASS_NAMES:
            break
    if not CLASS_NAMES:
        for js in sorted(INPUT.rglob('*.json')):
            try:
                data = json.loads(js.read_text(errors='ignore'))
            except Exception:
                continue
            if isinstance(data, dict) and isinstance(data.get('categories'), list) and data['categories']:
                cats = sorted(data['categories'], key=lambda c: c['id'])
                CLASS_NAMES = [c['name'] for c in cats]
                print(f'Class names from COCO categories in {js} (YOLO class i = i-th category by id) — please check the order below')
                break

    used_ids = set()
    for f, _ in pairs:
        for line in f.read_text(errors='ignore').splitlines():
            if line.split():
                used_ids.add(int(float(line.split()[0])))
    n_classes = max(used_ids) + 1 if used_ids else 0
    if not CLASS_NAMES or len(CLASS_NAMES) < n_classes:
        print(f'WARNING: no complete class-name list found ({len(CLASS_NAMES or [])} names for {n_classes} ids); using class_<id>.')
        CLASS_NAMES = [(CLASS_NAMES[i] if CLASS_NAMES and i < len(CLASS_NAMES) else f'class_{i}') for i in range(n_classes)]

    # Splits: keep the dataset's own split if it has one; otherwise a grouped random split.
    existing = Counter(split_of(f) or split_of(i) for f, i in pairs)
    if existing.get('train') and existing.get('val'):
        assignment = {f: (split_of(f) or split_of(i)) for f, i in pairs}
        print('Using the dataset\'s own split:', dict(existing))
        groups_per_split = defaultdict(set)
        for f, i in pairs:
            groups_per_split[assignment[f]].add(group_key(i.stem))
        leak = groups_per_split['train'] & (groups_per_split['val'] | groups_per_split['test'])
        if leak:
            print(f'WARNING: {len(leak)} photos appear (as augmented copies) in train AND val/test; test scores may be optimistic.')
    else:
        groups = defaultdict(list)
        for f, i in pairs:
            groups[group_key(i.stem)].append(f)
        keys = sorted(groups)
        random.Random(SEED).shuffle(keys)
        n_test = round(len(keys) * TEST_FRACTION)
        n_val = round(len(keys) * VAL_FRACTION)
        split_keys = {'test': set(keys[:n_test]), 'val': set(keys[n_test:n_test + n_val])}
        assignment = {}
        for k, files in groups.items():
            s = 'test' if k in split_keys['test'] else 'val' if k in split_keys['val'] else 'train'
            for f in files:
                assignment[f] = s
        print(f'No split in the dataset: grouped random split of {len(keys)} photos ({len(pairs)} images incl. augmented copies)')

    data_root = WORK / 'data'
    if data_root.exists():
        shutil.rmtree(data_root)
    for idx, (f, img) in enumerate(sorted(pairs, key=lambda p: str(p[1]))):
        s = assignment[f]
        name = f'{idx:05d}_{img.stem}'
        (data_root / 'images' / s).mkdir(parents=True, exist_ok=True)
        (data_root / 'labels' / s).mkdir(parents=True, exist_ok=True)
        os.symlink(img.resolve(), data_root / 'images' / s / f'{name}{img.suffix.lower()}')
        shutil.copyfile(f, data_root / 'labels' / s / f'{name}.txt')
    SPLITS = {s: str(data_root / 'images' / s) for s in ('train', 'val', 'test') if (data_root / 'images' / s).exists()}

# Write the data.yaml that training uses (absolute paths, in the writable working folder).
cfg = {'path': str(WORK), 'names': {i: n for i, n in enumerate(CLASS_NAMES)}, **SPLITS}
DATA = WORK / 'food_data.yaml'
YAML.save(DATA, cfg)
EVAL_SPLIT = 'test' if 'test' in SPLITS else 'val'
print(f'\n{len(CLASS_NAMES)} classes: {CLASS_NAMES}')
print(f'Splits: {SPLITS}\nEvaluation split: {EVAL_SPLIT}\nWrote {DATA}')

In [ ]:
# ---------------------------------------------------------------- dataset statistics ----
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image as ShowImage, display

def label_files(split_dir):
    d = Path(str(split_dir).replace(f'{os.sep}images', f'{os.sep}labels'))
    return list(d.rglob('*.txt'))

counts = {s: Counter() for s in SPLITS}
n_images = {}
for s, d in SPLITS.items():
    imgs = [p for p in Path(d).rglob('*') if p.suffix.lower() in IMG_EXT]
    n_images[s] = len(imgs)
    for lf in label_files(d):
        for line in lf.read_text(errors='ignore').splitlines():
            if line.split():
                counts[s][int(float(line.split()[0]))] += 1
print('Images per split:', n_images)
print(f'{"class":<28}' + ''.join(f'{s:>8}' for s in SPLITS))
for i, n in enumerate(CLASS_NAMES):
    print(f'{n:<28}' + ''.join(f'{counts[s][i]:>8}' for s in SPLITS))

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(CLASS_NAMES))
w = 0.8 / max(1, len(SPLITS))
for k, s in enumerate(SPLITS):
    ax.bar(x + k * w, [counts[s][i] for i in range(len(CLASS_NAMES))], w, label=s)
ax.set_xticks(x + w * (len(SPLITS) - 1) / 2, CLASS_NAMES, rotation=60, ha='right')
ax.set_ylabel('instances'); ax.set_title('Instances per class'); ax.legend()
fig.tight_layout(); fig.savefig(WORK / 'class_counts.png', dpi=120); plt.close(fig)
display(ShowImage(str(WORK / 'class_counts.png')))

## 2. Train (time-boxed)
`time=TRAIN_HOURS` makes Ultralytics fit the learning-rate schedule to the time budget and stop at it.
On a T4 this is roughly 100+ epochs for ~1.7k images at 640 px.

In [ ]:
def load_model():
    for w in [MODEL_WEIGHTS, *FALLBACK_WEIGHTS]:
        try:
            m = YOLO(w)
            print('Loaded pretrained weights:', w)
            return m, w
        except Exception as e:
            print(f'Could not load {w}: {e}')
    arch = Path(MODEL_WEIGHTS).with_suffix('.yaml').name
    print(f'WARNING: no pretrained weights (Internet off?). Training {arch} from scratch — results will be much worse.')
    return YOLO(arch), arch

model, weights_used = load_model()
train_args = dict(
    data=str(DATA), imgsz=IMGSZ, batch=BATCH, device=DEVICE, seed=SEED, deterministic=False,
    project=str(WORK / 'runs'), name='food_seg', exist_ok=True, plots=True,
    patience=PATIENCE, cos_lr=True, workers=4, verbose=True,
)
if SMOKE_TEST:
    train_args.update(epochs=1, imgsz=128, batch=4, workers=0, plots=False)
else:
    train_args.update(epochs=300, time=TRAIN_HOURS, cache='ram', amp=True)

t0 = time.time()
model.train(**train_args)
print(f'Training took {(time.time() - t0) / 60:.1f} min')
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / 'weights' / 'best.pt'
shutil.copyfile(BEST, WORK / 'best.pt')
for png in ('results.png',):
    if (RUN_DIR / png).exists():
        display(ShowImage(str(RUN_DIR / png)))

## 3. Evaluate on the held-out split
Mask mAP50 and mAP50-95 overall and per class, plus the confusion matrix.

In [ ]:
best = YOLO(str(BEST))
val_imgsz = train_args['imgsz']
m = best.val(data=str(DATA), split=EVAL_SPLIT, imgsz=val_imgsz, batch=train_args['batch'], device=DEVICE,
             plots=True, project=str(WORK / 'runs'), name='eval', exist_ok=True)
EVAL_DIR = Path(m.save_dir)

summary = {
    'split': EVAL_SPLIT,
    'images': n_images.get(EVAL_SPLIT),
    'mask_map50': float(m.seg.map50),
    'mask_map50_95': float(m.seg.map),
    'box_map50': float(m.box.map50),
    'box_map50_95': float(m.box.map),
}
print(f"Mask mAP50 = {summary['mask_map50']:.3f}   Mask mAP50-95 = {summary['mask_map50_95']:.3f}   "
      f"(box mAP50 = {summary['box_map50']:.3f})")

rows = []
for i, c in enumerate(m.ap_class_index):
    p, r, ap50, ap = m.seg.class_result(i)
    rows.append({'class_id': int(c), 'class': CLASS_NAMES[int(c)], 'instances': int(counts.get(EVAL_SPLIT, Counter())[int(c)]),
                 'mask_precision': round(float(p), 4), 'mask_recall': round(float(r), 4),
                 'mask_ap50': round(float(ap50), 4), 'mask_ap50_95': round(float(ap), 4)})
missing = [CLASS_NAMES[i] for i in range(len(CLASS_NAMES)) if i not in set(int(c) for c in m.ap_class_index)]
if missing:
    print('Classes with no instances in the evaluation split (no AP):', missing)

import csv
with open(WORK / 'per_class_ap.csv', 'w', newline='') as fh:
    writer = csv.DictWriter(fh, fieldnames=list(rows[0].keys()) if rows else ['class'])
    writer.writeheader(); writer.writerows(rows)
print(f'\n{"class":<28}{"n":>6}{"P":>8}{"R":>8}{"AP50":>8}{"AP50-95":>9}')
for r in sorted(rows, key=lambda r: r['mask_ap50_95']):
    print(f"{r['class']:<28}{r['instances']:>6}{r['mask_precision']:>8.3f}{r['mask_recall']:>8.3f}{r['mask_ap50']:>8.3f}{r['mask_ap50_95']:>9.3f}")

cm = m.confusion_matrix.matrix
cm_names = CLASS_NAMES + ['background']
with open(WORK / 'confusion_matrix.csv', 'w', newline='') as fh:
    writer = csv.writer(fh)
    writer.writerow(['predicted \\ true'] + cm_names)
    for i, row in enumerate(cm):
        writer.writerow([cm_names[i]] + [int(v) for v in row])
for png in sorted(EVAL_DIR.glob('confusion_matrix*.png')) + sorted(EVAL_DIR.glob('*Mask*PR_curve*.png')):
    shutil.copyfile(png, WORK / png.name)
    display(ShowImage(str(png)))

# min_confidence for the app: the confidence that maximises the mean mask F1 on the evaluation split.
MIN_CONFIDENCE, conf_source = 0.5, 'default'
try:
    f1 = np.asarray(m.seg.f1_curve)
    px = np.asarray(m.seg.px)
    if f1.size and px.size:
        best_conf = float(px[int(np.argmax(f1.mean(0)))])
        MIN_CONFIDENCE = round(min(0.8, max(0.25, best_conf)), 3)
        conf_source = f'max mean mask F1 on {EVAL_SPLIT} split (raw {best_conf:.3f}, clamped to 0.25–0.80)'
except Exception as e:
    print('Could not read the F1 curve:', e)
print('min_confidence =', MIN_CONFIDENCE, '—', conf_source)

## 4. Sample predictions

In [ ]:
from PIL import Image as PILImage
eval_imgs = sorted(p for p in Path(SPLITS[EVAL_SPLIT]).rglob('*') if p.suffix.lower() in IMG_EXT)
random.Random(SEED).shuffle(eval_imgs)
picked = eval_imgs[:12]
SAMPLES_DIR = WORK / 'samples'
if SAMPLES_DIR.exists():
    shutil.rmtree(SAMPLES_DIR)
SAMPLES_DIR.mkdir()
tiles = []
for p in picked:
    res = best.predict(str(p), imgsz=IMGSZ, conf=MIN_CONFIDENCE, device=DEVICE, verbose=False)[0]
    plotted = res.plot()[:, :, ::-1]  # BGR → RGB
    out = SAMPLES_DIR / f'{p.stem}.jpg'
    PILImage.fromarray(plotted).save(out, quality=90)
    tiles.append(PILImage.fromarray(plotted))
if tiles:
    cols, size = 4, 384
    rows_n = (len(tiles) + cols - 1) // cols
    grid = PILImage.new('RGB', (cols * size, rows_n * size), 'white')
    for k, t in enumerate(tiles):
        t = t.copy(); t.thumbnail((size, size))
        grid.paste(t, ((k % cols) * size + (size - t.width) // 2, (k // cols) * size + (size - t.height) // 2))
    grid.save(WORK / 'sample_predictions.jpg', quality=90)
    display(ShowImage(str(WORK / 'sample_predictions.jpg')))
print(f'Saved {len(tiles)} sample predictions to {SAMPLES_DIR}')

## 5. Export to ONNX for onnxruntime-web, check it, and write `food_seg.json`

In [ ]:
import onnx
import onnxruntime as ort

onnx_tmp = best.export(format='onnx', imgsz=IMGSZ, dynamic=False, simplify=True, opset=17, nms=False)
ONNX_PATH = WORK / 'food_seg.onnx'
shutil.copyfile(onnx_tmp, ONNX_PATH)
onnx.checker.check_model(str(ONNX_PATH))

sess = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])
inp = sess.get_inputs()[0]
outputs = {o.name: list(o.shape) for o in sess.get_outputs()}
print('input', inp.name, inp.shape, '| outputs', outputs)

def letterbox(img, size=IMGSZ, color=114):
    """Same preprocessing the app must do: keep aspect ratio, pad to size×size with grey 114, RGB, /255, NCHW."""
    w, h = img.size
    r = min(size / w, size / h)
    nw, nh = round(w * r), round(h * r)
    canvas = PILImage.new('RGB', (size, size), (color, color, color))
    canvas.paste(img.resize((nw, nh), PILImage.BILINEAR), ((size - nw) // 2, (size - nh) // 2))
    x = np.asarray(canvas, dtype=np.float32) / 255.0
    return x.transpose(2, 0, 1)[None]

check_img = PILImage.open(picked[0]).convert('RGB') if picked else PILImage.new('RGB', (IMGSZ, IMGSZ))
out0 = sess.run(None, {inp.name: letterbox(check_img)})[0]
nc = len(CLASS_NAMES)
scores = out0[0, 4:4 + nc, :]
print(f'ONNX check: output0 {out0.shape} = 4 box + {nc} class scores + {out0.shape[1] - 4 - nc} mask coefficients; '
      f'max class score on a sample image = {scores.max():.3f}')

meta = {
    'name': 'food_seg',
    'version': MODEL_VERSION,
    'task': 'instance-segmentation',
    'architecture': Path(weights_used).stem,
    'framework': f'ultralytics {ultralytics.__version__}',
    'classes': CLASS_NAMES,
    'inputSize': [IMGSZ, IMGSZ],
    'inputName': inp.name,
    'inputLayout': 'NCHW',
    'colorOrder': 'RGB',
    'preprocess': 'letterbox to inputSize keeping aspect ratio, pad value 114, then pixel/255, then (x - mean) / std',
    'letterboxPadValue': 114,
    'mean': [0.0, 0.0, 0.0],
    'std': [1.0, 1.0, 1.0],
    'outputs': outputs,
    'outputFormat': 'output0: [1, 4 + nClasses + 32, anchors] (cx, cy, w, h in input pixels; class scores; mask coefficients). '
                    'output1: [1, 32, maskH, maskW] mask prototypes. No NMS in the graph — apply NMS in the app.',
    'maskCoefficients': int(out0.shape[1] - 4 - nc),
    'nms': False,
    'iouThreshold': 0.7,
    'min_confidence': MIN_CONFIDENCE,
    'min_confidence_source': conf_source,
    'metrics': summary,
    'dataset': {
        'name': 'PUST Cafeteria Food Image Dataset',
        'doi': '10.17632/fn6yhzjz83.2',
        'url': 'https://data.mendeley.com/datasets/fn6yhzjz83/2',
        'images': n_images,
    },
    'trainedAt': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'trainHoursBudget': TRAIN_HOURS,
    'smokeTest': SMOKE_TEST,
}
(WORK / 'food_seg.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False))
(WORK / 'metrics.json').write_text(json.dumps({**summary, 'per_class': rows, 'min_confidence': MIN_CONFIDENCE}, indent=2, ensure_ascii=False))
print(json.dumps({k: meta[k] for k in ('classes', 'inputSize', 'mean', 'std', 'min_confidence', 'version')}, ensure_ascii=False))

In [ ]:
# ---------------------------------------------------------------- tidy up + summary ----
if CLEANUP and (WORK / 'data').exists():
    shutil.rmtree(WORK / 'data')  # symlinks to the read-only input; not needed in the output
for f in sorted(WORK.iterdir()):
    size = sum(p.stat().st_size for p in f.rglob('*') if p.is_file()) if f.is_dir() else f.stat().st_size
    print(f'{f.name:<36}{size / 1e6:>10.2f} MB')
print(f"\nMask mAP50 {summary['mask_map50']:.3f} | Mask mAP50-95 {summary['mask_map50_95']:.3f} | min_confidence {MIN_CONFIDENCE}")
print(f'Total run time: {(time.time() - RUN_STARTED) / 60:.1f} min')
print('Download food_seg.onnx and food_seg.json and put them in public/models/ in the Nirapod repo.')